In [ ]:
from google.colab import drive

#mounting Google Drive to save outputs permanently
drive.mount('/content/drive')

import os

#setting the HuggingFace token so datasets can be downloaded
# HF_TOKEN not required: all models used here are public

#creating the thesis output folders on Drive if they do not exist yet
os.makedirs('/content/drive/MyDrive/rag-thesis/results', exist_ok=True)
os.makedirs('/content/drive/MyDrive/rag-thesis/models/saved_models', exist_ok=True)

print("Google Drive mounted and thesis folders ready.")

Mounted at /content/drive
Google Drive mounted and thesis folders ready.


In [ ]:
import os

#removing any previous clone to start clean
if os.path.exists('/content/retrieval-effects-claim-verification'):
    import shutil
    shutil.rmtree('/content/retrieval-effects-claim-verification')

#cloning the private repo using the GitHub token for authentication
!git clone https://github.com/pernillejorg/retrieval-effects-claim-verification.git

%cd retrieval-effects-claim-verification

#checking out the step 5 pipeline branch
!git checkout step2.1-baseline-model

#pulling the last updated
!git pull origin step2.1-baseline-model

print("Repository cloned and on step2.1-baseline-model branch.")

Cloning into 'rag-claim-verification'...
remote: Enumerating objects: 472, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 472 (delta 66), reused 65 (delta 28), pack-reused 368 (from 1)
Receiving objects: 100% (472/472), 220.11 KiB | 7.86 MiB/s, done.
Resolving deltas: 100% (264/264), done.
/content/retrieval-effects-claim-verification
Branch 'step2.1-baseline-model' set up to track remote branch 'step2.1-baseline-model' from 'origin'.
Switched to a new branch 'step2.1-baseline-model'
From https://github.com/pernillejorg/rag-claim-verification
 * branch            step2.1-baseline-model -> FETCH_HEAD
Already up to date.
Repository cloned and on step2.1-baseline-model branch.


In [ ]:
#pinning datasets to the required version first to avoid conflicts
!pip install "datasets==2.21.0" -q

#installing all remaining project requirements from the requirements file
!pip install -r requirements.txt -q

!pip install rank-bm25 sentence-transformers -q

print("All dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 21.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 129.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.6/120.6 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 133.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 137.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 134.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 910.8/910.8 kB 60.9 MB/s eta 0:00:00
   ━━━━━━━

In [ ]:
import datasets
print("datasets version:", datasets.__version__)

datasets version: 2.21.0


In [ ]:
import os, shutil
target = '/content/retrieval-effects-claim-verification/data/scifact_open/cache'
os.makedirs(target, exist_ok=True)
source = '/content/drive/MyDrive/rag-thesis/data/scifact_open/cache'
for f in os.listdir(source):
    shutil.copy(os.path.join(source, f), os.path.join(target, f))
    print("copied", f)

copied corpus_candidates.jsonl
copied claims_metadata.jsonl
copied claims.jsonl
copied corpus.jsonl


In [ ]:
import os, shutil

#staging all three seeds' models (seed 42 = original names, 123/7 = suffixed)
model_folders = [
    "baseline_scifact", "evidence_scifact",                    # seed 42
    "baseline_scifact_seed123", "evidence_scifact_seed123",    # seed 123
    "baseline_scifact_seed7", "evidence_scifact_seed7",        # seed 7
]
for m in model_folders:
    src = f'/content/drive/MyDrive/rag-thesis/models/saved_models/{m}'
    dst = f'/content/retrieval-effects-claim-verification/models/saved_models/{m}'
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("staged", m)
    else:
        print("WARNING missing:", m)

staged baseline_scifact
staged evidence_scifact
staged baseline_scifact_seed123
staged evidence_scifact_seed123
staged baseline_scifact_seed7
staged evidence_scifact_seed7


In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())

CUDA: True


In [ ]:
!mkdir -p results/step5_variance

# ===== SEED 123 =====
!python models/pipeline.py --dataset scifact \
  --model1_path models/saved_models/baseline_scifact_seed123 \
  --model2_path models/saved_models/evidence_scifact_seed123 \
  --output_path results/step5_variance/pipeline_scifact_seed123.json \
  --records_path results/step5_variance/records_scifact_seed123.json 2>&1 | tee results/step5_variance/log_scifact_seed123.txt

!python models/pipeline.py --dataset scifact_open \
  --model1_path models/saved_models/baseline_scifact_seed123 \
  --model2_path models/saved_models/evidence_scifact_seed123 \
  --output_path results/step5_variance/pipeline_scifact_open_seed123.json \
  --records_path results/step5_variance/records_scifact_open_seed123.json 2>&1 | tee results/step5_variance/log_scifact_open_seed123.txt

# ===== SEED 7 =====
!python models/pipeline.py --dataset scifact \
  --model1_path models/saved_models/baseline_scifact_seed7 \
  --model2_path models/saved_models/evidence_scifact_seed7 \
  --output_path results/step5_variance/pipeline_scifact_seed7.json \
  --records_path results/step5_variance/records_scifact_seed7.json 2>&1 | tee results/step5_variance/log_scifact_seed7.txt

!python models/pipeline.py --dataset scifact_open \
  --model1_path models/saved_models/baseline_scifact_seed7 \
  --model2_path models/saved_models/evidence_scifact_seed7 \
  --output_path results/step5_variance/pipeline_scifact_open_seed7.json \
  --records_path results/step5_variance/records_scifact_open_seed7.json 2>&1 | tee results/step5_variance/log_scifact_open_seed7.txt

Using device: cuda
Loading Model 1 (claim-only) from: models/saved_models/baseline_scifact_seed123
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6554.42it/s]
Loading Model 2 (claim+evidence) from: models/saved_models/evidence_scifact_seed123
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5874.25it/s]
The repository for allenai/scifact contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/allenai/scifact.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y
Generating train split: 100%|██████████| 5183/5183 [00:00<00:00, 17528.88 examples/s]
Loaded 300 claims and corpus of 5183 documents

Building BM25 retriever...
Building BM25 index over 5183 documents...
BM25 index built successfully.
Building dense retriever (one-time encoding)...
Loading dense retrieval model: sentence-transformers/all-mpnet-base-v2
Us

In [ ]:
import os, shutil

#bringing the existing seed-42 step 5 results from Drive so the variance cell can read them
for fname in ["step5_pipeline_scifact_thr0_5.json", "step5_pipeline_scifact_open_thr0_5.json"]:
    src = f'/content/drive/MyDrive/rag-thesis/results/{fname}'
    dst = f'/content/retrieval-effects-claim-verification/results/{fname}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print("brought seed-42 file:", fname)
    else:
        print("WARNING: seed-42 file missing in Drive:", fname)

brought seed-42 file: step5_pipeline_scifact_thr0_5.json
brought seed-42 file: step5_pipeline_scifact_open_thr0_5.json


In [ ]:
import os

#verifying all 6 files the variance cell needs actually exist before computing
needed = [
    "results/step5_pipeline_scifact_thr0_5.json",
    "results/step5_pipeline_scifact_open_thr0_5.json",
    "results/step5_variance/pipeline_scifact_seed123_thr0_5.json",
    "results/step5_variance/pipeline_scifact_seed7_thr0_5.json",
    "results/step5_variance/pipeline_scifact_open_seed123_thr0_5.json",
    "results/step5_variance/pipeline_scifact_open_seed7_thr0_5.json",
]
for f in needed:
    print("OK " if os.path.exists(f) else "MISSING ", f)

OK  results/step5_pipeline_scifact_thr0_5.json
OK  results/step5_pipeline_scifact_open_thr0_5.json
OK  results/step5_variance/pipeline_scifact_seed123_thr0_5.json
OK  results/step5_variance/pipeline_scifact_seed7_thr0_5.json
OK  results/step5_variance/pipeline_scifact_open_seed123_thr0_5.json
OK  results/step5_variance/pipeline_scifact_open_seed7_thr0_5.json


In [ ]:
import json
import numpy as np

#seed 42 = the EXISTING step 5 files (in results/), 123/7 = new (in results/step5_variance/)
seed_files = {
    "SciFact": {
        42:  "results/step5_pipeline_scifact_thr0_5.json",
        123: "results/step5_variance/pipeline_scifact_seed123_thr0_5.json",
        7:   "results/step5_variance/pipeline_scifact_seed7_thr0_5.json",
    },
    "SciFact-Open": {
        42:  "results/step5_pipeline_scifact_open_thr0_5.json",
        123: "results/step5_variance/pipeline_scifact_open_seed123_thr0_5.json",
        7:   "results/step5_variance/pipeline_scifact_open_seed7_thr0_5.json",
    },
}

conditions = ["no_retrieval", "bm25_roberta", "dense_roberta", "dense_reranked_roberta"]

variance_summary = {}
for dataset, files in seed_files.items():
    print(f"\n===== {dataset} =====")
    per_seed = {}
    for seed, path in files.items():
        with open(path) as f:
            per_seed[seed] = json.load(f)["metrics"]
    variance_summary[dataset] = {}
    for cond in conditions:
        f1s = [per_seed[s][cond]["macro_f1"] for s in [42, 123, 7]]
        mean, std = float(np.mean(f1s)), float(np.std(f1s))
        variance_summary[dataset][cond] = {
            "seed42": f1s[0], "seed123": f1s[1], "seed7": f1s[2],
            "mean": round(mean, 4), "std": round(std, 4),
        }
        print(f"{cond:26s}  42={f1s[0]:.4f}  123={f1s[1]:.4f}  7={f1s[2]:.4f}   ->  {mean:.4f} ± {std:.4f}")

#saving the variance summary
with open('results/step5_variance/variance_summary.json', 'w') as f:
    json.dump(variance_summary, f, indent=2)
print("\nSaved results/step5_variance/variance_summary.json")


===== SciFact =====
no_retrieval                42=0.5263  123=0.5403  7=0.5059   ->  0.5242 ± 0.0141
bm25_roberta                42=0.5127  123=0.4777  7=0.3899   ->  0.4601 ± 0.0516
dense_roberta               42=0.5583  123=0.5212  7=0.4755   ->  0.5183 ± 0.0339
dense_reranked_roberta      42=0.4879  123=0.4558  7=0.3866   ->  0.4434 ± 0.0423

===== SciFact-Open =====
no_retrieval                42=0.6219  123=0.5803  7=0.5791   ->  0.5938 ± 0.0199
bm25_roberta                42=0.5229  123=0.5232  7=0.4164   ->  0.4875 ± 0.0502
dense_roberta               42=0.5560  123=0.5292  7=0.4087   ->  0.4980 ± 0.0641
dense_reranked_roberta      42=0.5075  123=0.5058  7=0.3776   ->  0.4637 ± 0.0608

Saved results/step5_variance/variance_summary.json


In [ ]:
#saving to drive 
import shutil
shutil.copytree('/content/retrieval-effects-claim-verification/results',
    '/content/drive/MyDrive/rag-thesis/results', dirs_exist_ok=True)
print("results saved to Drive")

results saved to Drive
